# GPU Credit Scoring - Entraînement XGBoost GPU

**Dataset:** `dataset_scoring_benin_700k.csv` (700K lignes)
**Objectif:** Entraîner XGBoost sur GPU (Tesla T4), comparer CPU/GPU, exporter le modèle

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
import time
import json
import os

print('XGBoost version:', xgb.__version__)
print('GPU available:', xgb.get_xgb_params() if hasattr(xgb, 'get_xgb_params') else 'check device')

In [ ]:
# Charger le dataset
DATA_PATH = '/kaggle/input/gpu-challenge-benin-credit-scoring/dataset_scoring_benin_700k.csv'
df = pd.read_csv(DATA_PATH)
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(df.head())

In [ ]:
# Identifier la cible et les features
TARGET = 'defaut_paiement'  # À confirmer selon le dataset réel
ID_COL = 'merchant_id' if 'merchant_id' in df.columns else df.columns[0]

exclude = [ID_COL, TARGET]
features = [c for c in df.columns if c not in exclude]
print(f'Features ({len(features)}):', features)
print(f'Target: {TARGET}')
print(f'Target distribution:\n{df[TARGET].value_counts(normalize=True)}')

In [ ]:
# Nettoyage
df = df.dropna(subset=[TARGET])
for col in features:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('unknown')
    else:
        df[col] = df[col].fillna(df[col].median())

X = df[features]
y = df[TARGET]
print(f'After cleaning: {X.shape}')

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Entraînement CPU
model_cpu = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    tree_method='hist', device='cpu',
    eval_metric='auc', random_state=42
)
t0 = time.time()
model_cpu.fit(X_train, y_train)
cpu_time = time.time() - t0
cpu_auc = roc_auc_score(y_test, model_cpu.predict_proba(X_test)[:, 1])
print(f'CPU: {cpu_time:.1f}s, AUC: {cpu_auc:.4f}')

In [ ]:
# Entraînement GPU
model_gpu = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    tree_method='hist', device='cuda',
    eval_metric='auc', random_state=42
)
t0 = time.time()
model_gpu.fit(X_train, y_train)
gpu_time = time.time() - t0
gpu_auc = roc_auc_score(y_test, model_gpu.predict_proba(X_test)[:, 1])
speedup = cpu_time / gpu_time
print(f'GPU: {gpu_time:.1f}s, AUC: {gpu_auc:.4f}, Speedup: {speedup:.1f}x')

In [ ]:
# Export du modèle
model_gpu.save_model('xgboost_model.json')

# Feature order
with open('feature_order.json', 'w') as f:
    json.dump(features, f)

# Metadata
metadata = {
    'version': '1.0.0',
    'n_features': len(features),
    'train_auc': float(roc_auc_score(y_train, model_gpu.predict_proba(X_train)[:, 1])),
    'test_auc': float(gpu_auc),
    'xgboost_version': xgb.__version__,
    'gpu_trained': True,
    'benchmark': {
        'dataset_size': len(df),
        'cpu_time': round(cpu_time, 1),
        'gpu_time': round(gpu_time, 1),
        'speedup': round(speedup, 1),
        'cpu_auc': round(cpu_auc, 4),
        'gpu_auc': round(gpu_auc, 4),
        'gpu_type': 'NVIDIA Tesla T4',
        'xgboost_version': xgb.__version__
    }
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Model exported!')
print(json.dumps(metadata, indent=2))

In [ ]:
# SHAP
import shap
explainer = shap.TreeExplainer(model_gpu)
shap_values = explainer.shap_values(X_test.iloc[:100])
shap.summary_plot(shap_values, X_test.iloc[:100], show=False)
print('SHAP computed successfully')